# 6.11 · 独立成分分析 / Independent Component Analysis (ICA)

> **课程定位 / Where this fits**
> PCA/FA(6.8/6.10)找**不相关**(二阶统计)的成分。ICA 走得更远——找**统计独立**的成分。经典应用是**鸡尾酒会问题**: 几个麦克风录到的是多个声源的混合, ICA 能把它们**盲源分离**回各自独立的信号。关键洞察: 独立 ≠ 不相关, 而独立性可通过"**非高斯性**"来寻找。
> ICA finds statistically independent (not just uncorrelated) components — the classic blind source separation / cocktail-party problem. Independence is found by maximising non-Gaussianity.

> 💡 **面试相关 / Interview-relevant**
> - "ICA 与 PCA 区别" ★★★★★（独立 vs 不相关; 非高斯 vs 方差）
> - "ICA 为什么要求源非高斯" ★★★★★（高斯下不可辨识）
> - "ICA 的不确定性(顺序/幅度/符号不可定)" ★★★★
> - "为什么先白化(whitening)" ★★★

---

## 学习目标 / Learning Objectives
1. 盲源分离问题 + ICA 模型 $\mathbf{X}=\mathbf{AS}$。
2. 独立 vs 不相关; 为何靠**非高斯性**。
3. 鸡尾酒会问题: 分离混合信号。
4. ICA vs PCA 对比 + 固有不确定性。

## 目录 / TOC
1. [盲源分离与非高斯 ⭐](#1)
2. [🎵 数据: 混合信号 + 分离 ⭐](#2)
3. [ICA vs PCA ⭐](#3)
4. [不确定性 + 预处理](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 盲源分离与非高斯 ⭐ / Blind Source Separation & Non-Gaussianity

**模型**: $k$ 个独立源 $\mathbf{s}$ 被一个未知混合矩阵 $\mathbf{A}$ 线性混合成观测 $\mathbf{x}=\mathbf{As}$(如 $k$ 个麦克风录 $k$ 个说话人)。ICA 目标: **只看 $\mathbf{x}$**, 估出解混矩阵 $\mathbf{W}\approx\mathbf{A}^{-1}$, 恢复源 $\hat{\mathbf{s}}=\mathbf{Wx}$。

**为什么靠非高斯**(ICA 核心): 由中心极限定理, 多个独立变量的**混合比单个源更接近高斯**。所以反过来——**让恢复信号尽量"非高斯"**, 就最可能恢复出原始独立源。常用非高斯度量: 峰度(kurtosis)、负熵(negentropy, FastICA 用)。

**关键限制**: 若源本身是**高斯**的, ICA **无法分离**(高斯旋转不变, 不可辨识)。所以 **ICA 要求源非高斯**。这与 PCA 形成鲜明对比: PCA 只用二阶统计(协方差), ICA 用高阶统计(非高斯性)。


<a id="2"></a>
## 2. 数据: 混合信号 + 分离 ⭐ / Mixed Signals & Separation

造 3 个**独立**且**非高斯**的源信号(正弦、方波、锯齿), 用随机矩阵混合成 3 路"录音", 看 ICA 能否盲分离回来。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.decomposition import FastICA, PCA
sns.set_theme(style="whitegrid")

rng = np.random.default_rng(0)
t = np.linspace(0, 8, 2000)
s1 = np.sin(2*t)                          # 正弦
s2 = np.sign(np.sin(3*t))                 # 方波
s3 = signal.sawtooth(2*np.pi*t)           # 锯齿
S = np.c_[s1, s2, s3]
S += 0.2*rng.normal(size=S.shape)         # 少量噪声
S /= S.std(0)
A = np.array([[1,1,1],[0.5,2,1.0],[1.5,1,2]])   # 混合矩阵
X = S @ A.T                                # 观测=混合录音
print(f"源 S: {S.shape} (正弦/方波/锯齿, 独立且非高斯), 混合成 X: {X.shape}")

ica = FastICA(n_components=3, random_state=0, whiten="unit-variance")
S_ica = ica.fit_transform(X)               # 盲分离

fig, axes = plt.subplots(3, 3, figsize=(14, 5.5), sharex=True)
for i in range(3):
    axes[0,i].plot(t[:500], S[:500,i]); axes[0,i].set_title(f"源 {i+1}" if i else "原始源")
    axes[1,i].plot(t[:500], X[:500,i], color="gray"); axes[1,i].set_title("混合观测" if i==0 else "")
    axes[2,i].plot(t[:500], S_ica[:500,i], color="green"); axes[2,i].set_title("ICA 恢复" if i==0 else "")
plt.tight_layout(); plt.show()
print("ICA 从混合录音里盲分离出三个独立源(波形恢复; 顺序/幅度/符号可能不同, 见第4节)")


<a id="3"></a>
## 3. ICA vs PCA ⭐ / ICA vs PCA

同样的混合信号, PCA 只能找**不相关**的正交方向, **分不开**这些源(因为不相关≠独立)。直接对比恢复质量。


In [ ]:
pca = PCA(n_components=3)
S_pca = pca.fit_transform(X)

def best_corr(rec, truth):
    # 每个真实源与某个恢复分量的最大|相关|, 取平均(应对顺序/符号不定)
    C = np.abs(np.corrcoef(np.c_[rec, truth].T)[:3, 3:])
    return C.max(0).mean()

print(f"ICA 恢复源的平均|相关|: {best_corr(S_ica, S):.3f}  (接近1=成功分离)")
print(f"PCA 恢复源的平均|相关|: {best_corr(S_pca, S):.3f}  (低=只去相关, 没分开独立源)")

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].plot(t[:500], S_ica[:500,0], color="green"); axes[0].set_title("ICA 第1分量: 干净的单一源")
axes[1].plot(t[:500], S_pca[:500,0], color="orange"); axes[1].set_title("PCA 第1分量: 仍是多源混合")
plt.tight_layout(); plt.show()
print("PCA 的分量仍是源的混合(只保证正交/不相关); ICA 才真正分离出独立源")


<a id="4"></a>
## 4. 不确定性 + 预处理 / Ambiguities & Preprocessing

ICA 有**固有不可辨识性**(面试常问):
- **顺序不定**: 恢复源的排列顺序任意(没有"第一个源"的概念)。
- **幅度/符号不定**: 源乘个常数(含 −1)可被混合矩阵吸收 → 幅度和正负不可定。

**预处理**: FastICA 先**白化(whitening)**——用 PCA 把数据去相关并归一化方差。这把问题简化成"在白化空间里找一个正交旋转使各轴最非高斯", 大大加速收敛。所以 ICA 常被看作"PCA 白化 + 找非高斯旋转"。


In [ ]:
from scipy.stats import kurtosis
print("非高斯性(超额峰度, 高斯=0)— ICA 的指南针:")
print(f"  原始源       峰度: {kurtosis(S, axis=0).round(2)}  (非0→非高斯, 可分离)")
print(f"  混合观测      峰度: {kurtosis(X, axis=0).round(2)}  (更接近0→混合更高斯, CLT)")
print(f"  ICA 恢复     峰度: {kurtosis(S_ica, axis=0).round(2)}  (重新变非高斯→找回了源)")
print("\nICA 通过'最大化非高斯性'找回源; 若源本身是高斯(峰度≈0)则无法分离")


<a id="5"></a>
## 5. 小结 / Summary

```
ICA: 盲源分离 x=As, 估解混 W≈A⁻¹ 恢复独立源 ŝ=Wx (鸡尾酒会问题)
核心: 独立 ≠ 不相关; 靠最大化"非高斯性"(峰度/负熵)找独立源(CLT: 混合更高斯)
限制: 源必须非高斯(高斯不可辨识); 顺序/幅度/符号不可定
预处理: 先白化(PCA 去相关+归一方差) → 在白化空间找最非高斯的正交旋转
vs PCA: PCA 二阶/不相关/正交方差; ICA 高阶/独立/非高斯
```

### 💡 面试速查
1. **ICA 找统计独立成分**(PCA 只找不相关); 盲源分离
2. **靠非高斯性**(峰度/负熵); 混合比源更高斯(CLT), 故最大化非高斯=恢复源
3. **源必须非高斯**, 否则不可辨识; 顺序/幅度/符号不定
4. **先白化**(PCA), 再找非高斯旋转(FastICA)
5. 应用: 脑电/语音分离、去伪迹、特征提取

### 下一节
**6.12 t-SNE**——前面的降维多为线性/全局。t-SNE 是非线性、**保局部邻域**的可视化神器, 能把高维数据(MNIST)摊成一眼看清簇的 2D 图。
